<a href="https://colab.research.google.com/github/kjahan/nlp_tips_and_trick/blob/main/notebooks/T5_Paraphrasing_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paraphrasing using T5

`Paraphrase any question with T5 (Text-To-Text Transfer Transformer) — Pretrained model and training script provided`

Ref:
https://towardsdatascience.com/paraphrase-any-question-with-t5-text-to-text-transfer-transformer-pretrained-model-and-cbb9e35f1555



In [1]:
!pip install --quiet transformers==2.8.0

     |████████████████████████████████| 563 kB 29.1 MB/s 
     |████████████████████████████████| 132 kB 66.0 MB/s 
     |████████████████████████████████| 1.3 MB 63.0 MB/s 
     |████████████████████████████████| 880 kB 68.2 MB/s 
     |████████████████████████████████| 5.6 MB 64.2 MB/s 
     |████████████████████████████████| 79 kB 7.4 MB/s 
     |████████████████████████████████| 9.1 MB 56.6 MB/s 
     |████████████████████████████████| 139 kB 75.6 MB/s 
     |████████████████████████████████| 127 kB 75.8 MB/s 


## Imports

In [2]:
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer

## Random Seeding

In [3]:
def set_seed(seed):
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

set_seed(42)

## Load T5 paraphraser model

In [4]:
model = T5ForConditionalGeneration.from_pretrained('ramsrigouthamg/t5_paraphraser')
tokenizer = T5Tokenizer.from_pretrained('ramsrigouthamg/t5_paraphraser')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device: {}".format(device))
model = model.to(device)

Downloading:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/892M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/792k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

device: cuda


## Input question for para-phrasing

In [16]:
# sentence = "Which course should I take to get started in data science?"
# sentence = "What are the ingredients required to bake a perfect cake?"
# sentence = "What is the best possible approach to learn aeronautical engineering?"
# sentence = "Do apples taste better than oranges in general?"
# sentence = "When it comes to investing into a digital currency, what's the best crypto for beginners?"
# sentence = "what is the best way to invest in cryptocurrency"
# sentence = "What are the 3 main causes of inflation?"
sentence = "Is buying a house during a recession a good idea?"

## Generate paraphrased text

In [17]:
text =  "paraphrase: " + sentence + " </s>"

max_len = 256

encoding = tokenizer.encode_plus(text,pad_to_max_length=True, return_tensors="pt")
input_ids, attention_masks = encoding["input_ids"].to(device), encoding["attention_mask"].to(device)

# set top_k = 50 and set top_p = 0.95 and num_return_sequences = 3
beam_outputs = model.generate(
    input_ids=input_ids, attention_mask=attention_masks,
    do_sample=True,
    max_length=256,
    top_k=120,
    top_p=0.98,
    early_stopping=True,
    num_return_sequences=10
)


print("\nOriginal Question: {}\n".format(sentence))
print ("Paraphrased Questions: ")
final_outputs =[]
for beam_output in beam_outputs:
    sent = tokenizer.decode(beam_output, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    if sent.lower() != sentence.lower() and sent not in final_outputs:
        final_outputs.append(sent)

for i, final_output in enumerate(final_outputs):
    print("{}: {}".format(i, final_output))


Original Question: Is buying a house during a recession a good idea?

Paraphrased Questions: 
0: How would you rate buying a house during a downturn?
1: If a recession happens to a country, does buying a house during that time in a good way?
2: Why I should buy a house during a time of economic downturn?
3: Can the economy help us buy a house in a recession?
4: Is buying a house in recession a good idea?
5: Do you think buying a house during a recession is a good idea?
6: Is buying a house for a recession a good idea?
